# watchin-u: 1. Prepare the dataset (Colab)

Turns raw camera footage in Drive into frames, a pre-filled visit log, contact sheets and
**pre-labelled CVAT bundles**. After labelling it builds the training datasets.

**Part A** (now): footage -> frames -> visit log + contact sheets -> CVAT pre-labels
**Part B** (after labelling in CVAT and filling the visit log): datasets for training

Runtime: a GPU is only needed for pre-labelling (Runtime -> Change runtime type -> T4 GPU).
Footage of the plant stays in your own Drive and Colab session.

## Setup: Drive, code, install

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# ---- settings ------------------------------------------------------------------
DRIVE = "/content/drive/MyDrive/watchinu"  # project folder in your Drive (created if missing)
REPO = "nav0225/watchin-u"
BRANCH = "ccr-09c28bb1-bjcddo"
# Private repo? Add a Colab secret named GITHUB_TOKEN (key icon in the left bar, read-only
# token). Or upload the code as  DRIVE/code/watchin-u.zip  and it is used instead.

import os, pathlib, shutil, subprocess, zipfile

pathlib.Path(DRIVE).mkdir(parents=True, exist_ok=True)
CODE = pathlib.Path("/content/watchin-u")
if CODE.exists():
    shutil.rmtree(CODE)
code_zip = pathlib.Path(DRIVE) / "code" / "watchin-u.zip"
if code_zip.exists():
    with zipfile.ZipFile(code_zip) as zf:
        zf.extractall("/content/_code")
    root = next(p.parent for p in pathlib.Path("/content/_code").rglob("pyproject.toml"))
    shutil.move(str(root), CODE)
    print("code from", code_zip)
else:
    token = ""
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN") or ""
    except Exception:
        pass
    url = f"https://{token + '@' if token else ''}github.com/{REPO}.git"
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(CODE)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        msg = r.stderr.replace(token, "***") if token else r.stderr
        raise RuntimeError("git clone failed - private repo? add GITHUB_TOKEN or the code zip\n" + msg)
print(subprocess.run(["git", "-C", str(CODE), "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -e "/content/watchin-u[ml]"
%cd /content/watchin-u
!watchinu --help | head -5

## Part A1: unpack footage and extract frames

In [ ]:
FOOTAGE_ZIP = "/content/drive/MyDrive/2026-11-04-dataset.zip"  # the zip you uploaded
EVERY_S = 1.0       # 10-s clips: 1 frame/s. Long recordings / time-lapse: 5-10
CAMERA_ID = "cam01"
BAY = "far"         # lane for pre-filled clip rows: "far" (shed wall) or "near" (road side)
SESSION = "2026-11-04"  # one folder per recording session in Drive

LOCAL = pathlib.Path("/content/work") / SESSION
FOOTAGE = LOCAL / "footage"
FRAMES = LOCAL / "frames"
FOOTAGE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(FOOTAGE_ZIP) as zf:
    zf.extractall(FOOTAGE)  # zip keeps file times: used as clip start when names have none
!find {FOOTAGE} -type f | head -50

In [ ]:
# What is in the footage? (resolution, codec, length) - camera exports should be 2560x1440
import json as _json

for f in sorted(p for p in FOOTAGE.rglob("*") if p.suffix.lower() in {".mp4", ".mkv", ".avi", ".mov", ".ts"}):
    r = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
                        "stream=width,height,codec_name,avg_frame_rate:format=duration",
                        "-of", "json", str(f)], capture_output=True, text=True)
    info = _json.loads(r.stdout or "{}")
    st, fm = (info.get("streams") or [{}])[0], info.get("format", {})
    print(f"{f.name:40s} {st.get('width')}x{st.get('height')} {st.get('codec_name')} "
          f"{st.get('avg_frame_rate')} fps  {float(fm.get('duration', 0)):.1f}s")

In [ ]:
!watchinu extract --input {FOOTAGE} --out {FRAMES} --every-s {EVERY_S} --camera-id {CAMERA_ID} --start-time-from-mtime
!watchinu dedupe --manifest {FRAMES}/manifest.jsonl --config configs/camera01.yaml

## Part A2: visit log + contact sheets

`visit_log.csv` gets one row per clip with the times filled in (or the reviewed log from the
repo, `labels/<SESSION>/visit_log.csv`, when it exists). There is **one row per lane per clip**
(`bay_id` = `far` or `near`): copy a row for the other lane if both are occupied. Fill
**product** (`wire_rod`, `ingot`, ...), **layout** (`linear` / `criss_cross` / `mixed`),
**coil_count** (from the loading slip) and **stage** if it is constant over the clip
(`wire_rod`, `covering`, `covered`, `other_load`). Use the same **truck_ref** for one truck
seen in several clips.

The contact sheets show 4 frames per clip. **Paste them into the Claude chat** so the
geometry and templates can be checked without downloading the footage.

In [ ]:
OUT = pathlib.Path(DRIVE) / "sessions" / SESSION
OUT.mkdir(parents=True, exist_ok=True)
reviewed = CODE / "labels" / SESSION / "visit_log.csv"   # per-lane log reviewed in the repo
if reviewed.exists():
    shutil.copy(reviewed, OUT / "visit_log.csv")
    print("using reviewed visit log from the repo:", reviewed)
else:
    !watchinu visit-log-from-clips --manifest {FRAMES}/manifest.jsonl --bay {BAY} --out {OUT}/visit_log.csv
!watchinu contact-sheet --manifest {FRAMES}/manifest.jsonl --out {OUT}/contact_sheets
from IPython.display import Image, display

for p in sorted((OUT / "contact_sheets").glob("*.jpg"))[:2]:
    display(Image(str(p), width=1100))

## Part A3: calibration check

Draws the configured bay polygon, bed corners and slot centres on a real frame. Send the
image to Claude. It is used to re-fit `configs/camera01.yaml` (the current geometry is a first
estimate from a phone photo).

In [ ]:
import json as _json
from IPython.display import Image, display

recs = [_json.loads(l) for l in open(FRAMES / "manifest.jsonl")]
big = [r for r in recs if r["width"] >= 2560] or recs
sample = FRAMES / big[len(big) // 2]["image"]
for layout in ("linear", "criss_cross"):
    !watchinu render-config --config configs/camera01.yaml --image {sample} --out {OUT}/calib_{layout}.png --layout {layout}
display(Image(str(OUT / "calib_linear.png"), width=1100))

## Part A4: pre-label for CVAT

Proposed `coil` / `person` boxes make labelling *correcting* rather than drawing. In CVAT:
create a task with labels `coil` and `person`, upload `images.zip`, then
**Actions -> Upload annotations -> YOLO 1.1** with `prelabel.zip`.

Expect to fix: boxes spanning two coils, an occasional box on a blue truck panel, and
missed coils at the image edges. Once about 200 frames are corrected and a v0 detector is
trained (notebook 2), re-run with `WEIGHTS` set to that `best.pt`. The pre-labels get much better.

In [ ]:
# v0: no labels yet -> YOLOE prompt-free proposals + the BALCO wrap colour test (coils)
#     and a COCO model (workers).
# v1+: set WEIGHTS to our fine-tuned detector, e.g. f"{DRIVE}/runs/detect-v1/weights/best.pt"
WEIGHTS = None
CVAT = OUT / "cvat"
CVAT.mkdir(exist_ok=True)
method = f"--method weights --weights {WEIGHTS}" if WEIGHTS else "--method wrap-colour"
!watchinu prelabel {method} --manifest {FRAMES}/manifest.jsonl --out {CVAT}/prelabel.zip --images-zip {CVAT}/images.zip --device 0
!ls -la {CVAT}

In [ ]:
# Keep the frames in Drive (needed again in Part B and for later sessions)
shutil.copytree(FRAMES, OUT / "frames", dirs_exist_ok=True)
print("saved to", OUT)

## Part B: build training datasets (after labelling)

Needs, in `DRIVE/sessions/<SESSION>/`:
* `frames/` (from Part A),
* `visit_log.csv` **filled in**,
* `labels/`: unzip the CVAT export (**YOLO 1.1**) here; the `.txt` files are in `obj_train_data/`.

Multiple sessions: run Part A for each, then list them all in `SESSIONS` below.

In [ ]:
import json as _json

SESSIONS = ["2026-11-04"]      # every labelled session, e.g. ["2026-11-04", "2026-11-06"]
DATA = pathlib.Path(DRIVE) / "datasets"
merged = pathlib.Path("/content/merged")
if merged.exists():
    shutil.rmtree(merged)
(merged / "labels").mkdir(parents=True)
manifest_lines, logs = [], []
for sess in SESSIONS:
    src = pathlib.Path(DRIVE) / "sessions" / sess
    shutil.copytree(src / "frames" / "images", merged / "images", dirs_exist_ok=True)
    manifest_lines += open(src / "frames" / "manifest.jsonl").read().splitlines()
    lbl = src / "labels" / "obj_train_data"
    if lbl.exists():
        for f in lbl.glob("*.txt"):
            shutil.copy(f, merged / "labels" / f.name)
    logs.append(open(src / "visit_log.csv").read().splitlines())
(merged / "manifest.jsonl").write_text("\n".join(manifest_lines) + "\n")
header = logs[0][0]
(merged / "visit_log.csv").write_text("\n".join([header] + [l for lg in logs for l in lg[1:]]) + "\n")
print(len(manifest_lines), "frames,", sum(len(l) - 1 for l in logs), "visits")

In [ ]:
# Label audit BEFORE training: coils per lane and recording vs the visit log.
# Rows flagged "coils in an empty lane", "more coils than coil_count", ... need a look in CVAT.
!watchinu label-audit --manifest {merged}/manifest.jsonl --labels {merged}/labels --visit-log {merged}/visit_log.csv --config configs/camera01.yaml

In [ ]:
M = merged / "manifest.jsonl"
V = merged / "visit_log.csv"
FROZEN = pathlib.Path(DRIVE) / "gold_test_visits.txt"   # created on the first run, then kept
# "visit" is leakage-safe. With a single session where trucks overlap all day, everything
# is one group: use "clip" ONCE to bootstrap v0 (val scores optimistic, test disabled).
GROUP_BY = "visit" if len(SESSIONS) > 1 else "clip"
TEST = 0.2 if GROUP_BY == "visit" else 0.0
frozen_arg = f"--frozen-test {FROZEN}" if FROZEN.exists() else ""
!watchinu split --manifest {M} --visit-log {V} --val 0.2 --test {TEST} --group-by {GROUP_BY} {frozen_arg}
if GROUP_BY == "visit" and not FROZEN.exists():
    rows = [_json.loads(line) for line in open(M)]
    test_visits = sorted({r["visit_id"] for r in rows if r["split"] == "test" and r["visit_id"]})
    FROZEN.write_text("\n".join(test_visits) + "\n")
    print("froze", len(test_visits), "gold test visits ->", FROZEN)

# detector: full frames (as at runtime); classifiers: one crop per lane with lane-specific labels
!watchinu export-detect --manifest {M} --labels {merged}/labels --out /content/ds/detect --classes coil,person --config configs/camera01.yaml
!watchinu export-cls --manifest {M} --visit-log {V} --task stage --out /content/ds/stage --config configs/camera01.yaml
!watchinu export-cls --manifest {M} --visit-log {V} --task layout --out /content/ds/layout --config configs/camera01.yaml
DATA.mkdir(parents=True, exist_ok=True)
for name in ("detect", "stage", "layout"):
    shutil.make_archive(str(DATA / name), "zip", f"/content/ds/{name}")
!ls -la {DATA}